# Kaggriculture: V7 Replay Diagnosis and V8 Candidate Selection

## TL;DR

The most useful replay losses were not the largest blowouts. Three losses had at least **90% mean field-action similarity** and finished only **1.3k–2.9k coins** behind, making them unusually clean evidence for market-timing differences.

The full 71-episode baseline corpus then revealed a broader shift: many later losses clustered near 80% field similarity, so a small market-only patch was unlikely to be sufficient. Moon V9 covered the historical replay tapes well, but failed against several newer live policies. Beicicc C18 was more robust across both gates.

This notebook reports aggregated experiments only. It contains no agent source, action trace, opponent identity, episode ID, random seed, submission ID, or leaderboard rating.

## Context & Methods

Kaggriculture is a two-player, 720-turn farming simulation with public farm state and a shared dynamic market. Each action has three components: `farmer`, `hands`, and `market`.

This analysis uses three complementary views:

1. **Observed replay diagnosis** — compare the two agents' actions in completed public episodes.
2. **Frozen-tape counterfactual** — replay the original configuration while replacing one policy and keeping the opponent's recorded action sequence fixed.
3. **Live two-seat cross-play** — run both policies from observations on unseen seeds, once in each `player` position.

### Key assumptions and caveats

- Action similarity means exact equality of the recorded action component on a turn.
- Field similarity is the mean of `farmer` and `hands` similarity.
- A strict near-mirror loss has field similarity of at least 90% and a negative final coin margin.
- Frozen-tape results are compatibility tests, not estimates of future leaderboard win rate. The recorded opponent cannot react to a changed market.
- Live cross-play is therefore a required second gate.
- The embedded records come from public replay artifacts made downloadable by the competition environment. Only de-identified aggregates are published here.

## Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

# Aggregated, de-identified records derived from completed public Kaggle replays.
# No opponent names, episode identifiers, seeds, submission identifiers,
# private observations, agent source, or action traces are included.
corpus = pd.DataFrame([{'margin': 129166.0, 'field_similarity': 0.0924895688456189, 'farmer_similarity': 0.1363004172461752, 'hands_similarity': 0.0486787204450625, 'market_similarity': 0.1655076495132127, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 82434.0, 'field_similarity': 0.0639777468706536, 'farmer_similarity': 0.086230876216968, 'hands_similarity': 0.0417246175243393, 'market_similarity': 0.1349095966620306, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 14538.0, 'field_similarity': 0.9019471488178024, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.8178025034770514, 'market_similarity': 0.4325452016689847, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 133441.0, 'field_similarity': 0.0827538247566063, 'farmer_similarity': 0.1237830319888734, 'hands_similarity': 0.0417246175243393, 'market_similarity': 0.1655076495132127, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 44898.0, 'field_similarity': 0.0834492350486787, 'farmer_similarity': 0.125173852573018, 'hands_similarity': 0.0417246175243393, 'market_similarity': 0.1404728789986092, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 78107.0, 'field_similarity': 0.105702364394993, 'farmer_similarity': 0.1613351877607788, 'hands_similarity': 0.0500695410292072, 'market_similarity': 0.1349095966620306, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 32976.0, 'field_similarity': 0.0757997218358831, 'farmer_similarity': 0.1098748261474269, 'hands_similarity': 0.0417246175243393, 'market_similarity': 0.1474269819193324, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 24676.0, 'field_similarity': 0.086230876216968, 'farmer_similarity': 0.1307371349095966, 'hands_similarity': 0.0417246175243393, 'market_similarity': 0.1321279554937413, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 13476.0, 'field_similarity': 0.1112656467315716, 'farmer_similarity': 0.1543810848400556, 'hands_similarity': 0.0681502086230876, 'market_similarity': 0.0598052851182197, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 5334.0, 'field_similarity': 0.0897079276773296, 'farmer_similarity': 0.1376912378303199, 'hands_similarity': 0.0417246175243393, 'market_similarity': 0.1196105702364394, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 3855.0, 'field_similarity': 0.956884561891516, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.9276773296244784, 'market_similarity': 0.43115438108484, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 1027.0, 'field_similarity': 0.9235048678720446, 'farmer_similarity': 0.9680111265646733, 'hands_similarity': 0.8789986091794159, 'market_similarity': 0.3282336578581363, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 3575.0, 'field_similarity': 0.9853963838664812, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.9847009735744088, 'market_similarity': 0.43115438108484, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': -1733.0, 'field_similarity': 0.9603616133518776, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.9346314325452016, 'market_similarity': 0.4005563282336578, 'result': 'loss', 'strict_near_mirror_loss': True}, {'margin': 510.0, 'field_similarity': 0.9596662030598052, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.933240611961057, 'market_similarity': 0.3282336578581363, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': -1276.0, 'field_similarity': 0.9582753824756608, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.9304589707927676, 'market_similarity': 0.3268428372739916, 'result': 'loss', 'strict_near_mirror_loss': True}, {'margin': 13110.0, 'field_similarity': 0.258692628650904, 'farmer_similarity': 0.4464534075104311, 'hands_similarity': 0.0709318497913769, 'market_similarity': 0.1349095966620306, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 178.0, 'field_similarity': 0.972183588317107, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.9582753824756608, 'market_similarity': 0.3184979137691238, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': -7483.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.325452016689847, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': 9330.0, 'field_similarity': 0.97009735744089, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.9541029207232268, 'market_similarity': 0.4339360222531293, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 14349.0, 'field_similarity': 0.97009735744089, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.9541029207232268, 'market_similarity': 0.4339360222531293, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 132.0, 'field_similarity': 0.972183588317107, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.9582753824756608, 'market_similarity': 0.3282336578581363, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 8150.0, 'field_similarity': 0.2927677329624478, 'farmer_similarity': 0.4826147426981919, 'hands_similarity': 0.1029207232267037, 'market_similarity': 0.1766342141863699, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 11513.0, 'field_similarity': 0.97009735744089, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.9541029207232268, 'market_similarity': 0.4339360222531293, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 14628.0, 'field_similarity': 0.9853963838664812, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.9847009735744088, 'market_similarity': 0.4353268428372739, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 13259.0, 'field_similarity': 0.9707927677329624, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.9554937413073712, 'market_similarity': 0.4297635605006954, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 10918.0, 'field_similarity': 0.97009735744089, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.9541029207232268, 'market_similarity': 0.4242002781641168, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 15358.0, 'field_similarity': 0.9853963838664812, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.9847009735744088, 'market_similarity': 0.43115438108484, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 13010.0, 'field_similarity': 0.9853963838664812, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.9847009735744088, 'market_similarity': 0.43115438108484, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 12525.0, 'field_similarity': 0.982614742698192, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.9791376912378305, 'market_similarity': 0.4353268428372739, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 12273.0, 'field_similarity': 0.97009735744089, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.9541029207232268, 'market_similarity': 0.4339360222531293, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 13700.0, 'field_similarity': 0.97009735744089, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.9541029207232268, 'market_similarity': 0.4339360222531293, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 15179.0, 'field_similarity': 0.97009735744089, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.9541029207232268, 'market_similarity': 0.4339360222531293, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 14258.0, 'field_similarity': 0.97009735744089, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.9541029207232268, 'market_similarity': 0.4339360222531293, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 337.0, 'field_similarity': 0.972183588317107, 'farmer_similarity': 0.9860917941585536, 'hands_similarity': 0.9582753824756608, 'market_similarity': 0.3296244784422809, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': -2936.0, 'field_similarity': 0.9109874826147428, 'farmer_similarity': 0.9596662030598052, 'hands_similarity': 0.8623087621696801, 'market_similarity': 0.2712100139082058, 'result': 'loss', 'strict_near_mirror_loss': True}, {'margin': -6639.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.325452016689847, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -4950.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.2739916550764951, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -10855.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.325452016689847, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -13831.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.325452016689847, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -16142.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.325452016689847, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -4732.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.325452016689847, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -6157.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.325452016689847, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -6276.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.325452016689847, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': 1841.0, 'field_similarity': 0.8567454798331016, 'farmer_similarity': 0.8692628650904033, 'hands_similarity': 0.8442280945757997, 'market_similarity': 0.3518776077885953, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': -7567.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.325452016689847, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -7123.0, 'field_similarity': 0.8136300417246175, 'farmer_similarity': 0.8692628650904033, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.3268428372739916, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -4359.0, 'field_similarity': 0.8004172461752433, 'farmer_similarity': 0.8692628650904033, 'hands_similarity': 0.7315716272600834, 'market_similarity': 0.2336578581363004, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -7761.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.3157162726008345, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -2488.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.325452016689847, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': 1350.0, 'field_similarity': 0.8052851182197497, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7357440890125174, 'market_similarity': 0.3157162726008345, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': 1868.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.2739916550764951, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': -8063.0, 'field_similarity': 0.8087621696801113, 'farmer_similarity': 0.8720445062586927, 'hands_similarity': 0.7454798331015299, 'market_similarity': 0.3129346314325452, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -8207.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.325452016689847, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -2993.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.3087621696801113, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -4207.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.3087621696801113, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -6481.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.325452016689847, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -5604.0, 'field_similarity': 0.8031988873435327, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7315716272600834, 'market_similarity': 0.2503477051460361, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -2507.0, 'field_similarity': 0.8031988873435327, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7315716272600834, 'market_similarity': 0.2600834492350486, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -8831.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.3073713490959666, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -6775.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.3073713490959666, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': 627.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.3087621696801113, 'result': 'win', 'strict_near_mirror_loss': False}, {'margin': -7566.0, 'field_similarity': 0.8031988873435327, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7315716272600834, 'market_similarity': 0.2336578581363004, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -8062.0, 'field_similarity': 0.8031988873435327, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7315716272600834, 'market_similarity': 0.2600834492350486, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -6248.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.325452016689847, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -7709.0, 'field_similarity': 0.8150208623087623, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7552155771905424, 'market_similarity': 0.321279554937413, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -5656.0, 'field_similarity': 0.8031988873435327, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7315716272600834, 'market_similarity': 0.2600834492350486, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -5713.0, 'field_similarity': 0.8164116828929069, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7579972183588317, 'market_similarity': 0.2503477051460361, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -5783.0, 'field_similarity': 0.8031988873435327, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7315716272600834, 'market_similarity': 0.2600834492350486, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -10679.0, 'field_similarity': 0.8031988873435327, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7315716272600834, 'market_similarity': 0.258692628650904, 'result': 'loss', 'strict_near_mirror_loss': False}, {'margin': -5642.0, 'field_similarity': 0.8025034770514603, 'farmer_similarity': 0.874826147426982, 'hands_similarity': 0.7301808066759388, 'market_similarity': 0.2600834492350486, 'result': 'loss', 'strict_near_mirror_loss': False}])
moon_counterfactual = pd.DataFrame([{'margin_before': 129166.0, 'margin_after': 127457.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 82434.0, 'margin_after': 134331.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 14538.0, 'margin_after': 20158.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 133441.0, 'margin_after': 137372.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 44898.0, 'margin_after': 49962.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 78107.0, 'margin_after': 75457.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 32976.0, 'margin_after': 36959.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 24676.0, 'margin_after': 30300.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 13476.0, 'margin_after': 23383.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 5334.0, 'margin_after': 9015.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 3855.0, 'margin_after': 22771.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 1027.0, 'margin_after': 15995.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 3575.0, 'margin_after': 23698.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': -1733.0, 'margin_after': 5864.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': 510.0, 'margin_after': 8046.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': -1276.0, 'margin_after': 9573.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': 13110.0, 'margin_after': 12014.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 178.0, 'margin_after': 7897.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': -7483.0, 'margin_after': 178459.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': 9330.0, 'margin_after': 24144.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 14349.0, 'margin_after': 22149.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 132.0, 'margin_after': 5361.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 8150.0, 'margin_after': 7443.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 11513.0, 'margin_after': 20791.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 14628.0, 'margin_after': 18005.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 13259.0, 'margin_after': 23250.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 10918.0, 'margin_after': 21186.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 15358.0, 'margin_after': 16849.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 13010.0, 'margin_after': 20680.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 12525.0, 'margin_after': 18904.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 12273.0, 'margin_after': 22644.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 13700.0, 'margin_after': 19700.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 15179.0, 'margin_after': 21725.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 14258.0, 'margin_after': 23955.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 337.0, 'margin_after': 5821.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': -2936.0, 'margin_after': 16400.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -6639.0, 'margin_after': 163980.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -4950.0, 'margin_after': 1590.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -10855.0, 'margin_after': 155047.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -13831.0, 'margin_after': 152866.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -16142.0, 'margin_after': 173390.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -4732.0, 'margin_after': 179252.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -6157.0, 'margin_after': 182162.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -6276.0, 'margin_after': 182894.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': 1841.0, 'margin_after': 7711.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': -7567.0, 'margin_after': 173460.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -7123.0, 'margin_after': 144965.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -4359.0, 'margin_after': 165.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -7761.0, 'margin_after': 7010.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -2488.0, 'margin_after': 162457.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': 1350.0, 'margin_after': 7046.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 1868.0, 'margin_after': 6403.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': -8063.0, 'margin_after': 4652.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -8207.0, 'margin_after': 179488.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -2993.0, 'margin_after': 168892.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -4207.0, 'margin_after': 171446.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -6481.0, 'margin_after': 162488.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -5604.0, 'margin_after': -2548.0, 'result_before': 'loss', 'result_after': 'loss'}, {'margin_before': -2507.0, 'margin_after': 3372.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -8831.0, 'margin_after': 158119.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -6775.0, 'margin_after': 169772.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': 627.0, 'margin_after': 173258.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': -7566.0, 'margin_after': 3003.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -8062.0, 'margin_after': 4086.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -6248.0, 'margin_after': 174359.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -7709.0, 'margin_after': 158098.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -5656.0, 'margin_after': 5782.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -5713.0, 'margin_after': -3560.0, 'result_before': 'loss', 'result_after': 'loss'}, {'margin_before': -5783.0, 'margin_after': 4688.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -10679.0, 'margin_after': 1468.0, 'result_before': 'loss', 'result_after': 'win'}])
c18_counterfactual = pd.DataFrame([{'margin_before': 129166.0, 'margin_after': 126547.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 82434.0, 'margin_after': 130833.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 14538.0, 'margin_after': 16339.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 133441.0, 'margin_after': 135324.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 44898.0, 'margin_after': 48204.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 78107.0, 'margin_after': 74661.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 32976.0, 'margin_after': 35337.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 24676.0, 'margin_after': 28948.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 13476.0, 'margin_after': 23211.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 5334.0, 'margin_after': 14089.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 3855.0, 'margin_after': 19067.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 1027.0, 'margin_after': 13760.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 3575.0, 'margin_after': 20164.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': -1733.0, 'margin_after': 3949.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': 510.0, 'margin_after': 6733.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': -1276.0, 'margin_after': 7482.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': 13110.0, 'margin_after': 9299.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 178.0, 'margin_after': 6789.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': -7483.0, 'margin_after': 175745.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': 9330.0, 'margin_after': 19878.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 14349.0, 'margin_after': 16820.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 132.0, 'margin_after': 4436.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 8150.0, 'margin_after': 4417.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 11513.0, 'margin_after': 15986.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 14628.0, 'margin_after': 13799.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 13259.0, 'margin_after': 18012.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 10918.0, 'margin_after': 18101.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 15358.0, 'margin_after': 13932.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 13010.0, 'margin_after': 17479.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 12525.0, 'margin_after': 15346.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 12273.0, 'margin_after': 17565.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 13700.0, 'margin_after': 17019.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 15179.0, 'margin_after': 17324.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 14258.0, 'margin_after': 20169.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 337.0, 'margin_after': 4589.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': -2936.0, 'margin_after': 14256.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -6639.0, 'margin_after': 162088.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -4950.0, 'margin_after': 2368.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -10855.0, 'margin_after': 154018.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -13831.0, 'margin_after': 145040.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -16142.0, 'margin_after': 171159.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -4732.0, 'margin_after': 168039.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -6157.0, 'margin_after': 178951.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -6276.0, 'margin_after': 180743.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': 1841.0, 'margin_after': 7016.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': -7567.0, 'margin_after': 171325.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -7123.0, 'margin_after': 141564.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -4359.0, 'margin_after': 4856.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -7761.0, 'margin_after': 5722.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -2488.0, 'margin_after': 148614.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': 1350.0, 'margin_after': 5530.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': 1868.0, 'margin_after': 3207.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': -8063.0, 'margin_after': 3221.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -8207.0, 'margin_after': 176861.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -2993.0, 'margin_after': 166019.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -4207.0, 'margin_after': 169730.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -6481.0, 'margin_after': 154876.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -5604.0, 'margin_after': 4805.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -2507.0, 'margin_after': 5294.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -8831.0, 'margin_after': 155352.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -6775.0, 'margin_after': 165508.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': 627.0, 'margin_after': 170478.0, 'result_before': 'win', 'result_after': 'win'}, {'margin_before': -7566.0, 'margin_after': 6327.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -8062.0, 'margin_after': 4319.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -6248.0, 'margin_after': 171897.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -7709.0, 'margin_after': 150390.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -5656.0, 'margin_after': 5110.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -5713.0, 'margin_after': 4478.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -5783.0, 'margin_after': 4383.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -10679.0, 'margin_after': 2447.0, 'result_before': 'loss', 'result_after': 'win'}, {'margin_before': -5642.0, 'margin_after': 5464.0, 'result_before': 'loss', 'result_after': 'win'}])
moon_crossplay = pd.DataFrame([{'opponent': 'Soil V8', 'games': 4, 'wins': 4, 'ties': 0, 'losses': 0, 'mean_margin': 3488.0, 'worst_margin': 442.0}, {'opponent': 'Rayk C18', 'games': 4, 'wins': 4, 'ties': 0, 'losses': 0, 'mean_margin': 167666.0, 'worst_margin': 154086.0}, {'opponent': 'Kaito V18', 'games': 4, 'wins': 0, 'ties': 0, 'losses': 4, 'mean_margin': -3404.5, 'worst_margin': -6074.0}, {'opponent': 'Hamburger', 'games': 4, 'wins': 0, 'ties': 0, 'losses': 4, 'mean_margin': -1771.0, 'worst_margin': -2166.0}, {'opponent': 'Beicicc C18', 'games': 4, 'wins': 0, 'ties': 0, 'losses': 4, 'mean_margin': -3995.5, 'worst_margin': -7150.0}, {'opponent': 'Adaptive visible-state', 'games': 4, 'wins': 4, 'ties': 0, 'losses': 0, 'mean_margin': 170548.0, 'worst_margin': 163254.0}, {'opponent': 'Blacklions V19', 'games': 4, 'wins': 4, 'ties': 0, 'losses': 0, 'mean_margin': 168572.75, 'worst_margin': 158412.0}])
c18_crossplay = pd.DataFrame([{'opponent': 'Kaito V18', 'games': 6, 'wins': 4, 'ties': 2, 'losses': 0, 'mean_margin': 401.0, 'worst_margin': 0.0}, {'opponent': 'Hamburger', 'games': 6, 'wins': 6, 'ties': 0, 'losses': 0, 'mean_margin': 4915.833333333333, 'worst_margin': 1664.0}, {'opponent': 'Moon V9', 'games': 6, 'wins': 6, 'ties': 0, 'losses': 0, 'mean_margin': 4798.333333333333, 'worst_margin': 1429.0}, {'opponent': 'Soil V8', 'games': 6, 'wins': 6, 'ties': 0, 'losses': 0, 'mean_margin': 4279.833333333333, 'worst_margin': 1299.0}])

{
    "baseline_episodes": len(corpus),
    "baseline_wins": int(corpus["result"].eq("win").sum()),
    "baseline_losses": int(corpus["result"].eq("loss").sum()),
    "moon_counterfactual_episodes": len(moon_counterfactual),
    "c18_counterfactual_episodes": len(c18_counterfactual),
    "strong_family_live_games": int(c18_crossplay["games"].sum()),
}

The baseline snapshot contains 71 completed public episodes: 36 wins and 35 losses. The first specialist counterfactual uses the 70-episode snapshot available when that experiment was run; the selected closed-loop candidate uses the updated 71-episode snapshot. Results are not pooled across those two snapshot sizes.

## Results

### 1. Small-gap near mirrors isolate marginal differences

In [2]:
colors = np.where(corpus["result"].eq("win"), "#2f855a", "#c53030")
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(corpus["field_similarity"] * 100, corpus["margin"], c=colors, alpha=0.78)
ax.axhline(0, color="#222222", linewidth=1)
ax.axvline(90, color="#4a5568", linestyle="--", linewidth=1,
           label="90% near-mirror threshold")
highlight = corpus.loc[corpus["strict_near_mirror_loss"]]
ax.scatter(highlight["field_similarity"] * 100, highlight["margin"],
           facecolors="none", edgecolors="#1a202c", linewidths=1.5, s=115,
           label="Strict near-mirror losses")
ax.set(title="Final margin versus field-action similarity — 71 public episodes",
       xlabel="Mean farmer/hands action similarity (%)",
       ylabel="Final coin margin")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

highlight[["margin", "farmer_similarity", "hands_similarity", "market_similarity"]].assign(
    farmer_similarity=lambda frame: (100 * frame["farmer_similarity"]).round(1),
    hands_similarity=lambda frame: (100 * frame["hands_similarity"]).round(1),
    market_similarity=lambda frame: (100 * frame["market_similarity"]).round(1),
).rename(columns={
    "farmer_similarity": "farmer_similarity_pct",
    "hands_similarity": "hands_similarity_pct",
    "market_similarity": "market_similarity_pct",
}).sort_values("margin", ascending=False).reset_index(drop=True)

The strict near-mirror losses finish **1,276**, **1,733**, and **2,936** coins behind. Their field schedules are close enough that market timing becomes a credible marginal explanation. However, the dense cluster of later losses near 80% field similarity shows that the population also moved to different complete routes.

### 2. Moon V9: historical replay coverage can select the wrong live policy

In [3]:
moon_counterfactual["transition"] = (
    moon_counterfactual["result_before"] + "→" + moon_counterfactual["result_after"]
)
transition_order = ["win→win", "loss→win", "loss→loss", "win→loss"]
moon_transitions = (
    moon_counterfactual["transition"].value_counts()
    .reindex(transition_order, fill_value=0)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
moon_transitions.plot.bar(
    ax=axes[0], color=["#2f855a", "#38a169", "#c53030", "#9b2c2c"]
)
axes[0].set(title="Moon V9 on 70 frozen tapes",
            xlabel="", ylabel="Episodes")
axes[0].tick_params(axis="x", rotation=0)

ordered = moon_counterfactual.reset_index(drop=True)
axes[1].plot(ordered.index, ordered["margin_before"],
             label="Baseline observed", color="#718096")
axes[1].plot(ordered.index, ordered["margin_after"],
             label="Moon V9 counterfactual", color="#2b6cb0")
axes[1].axhline(0, color="#222222", linewidth=1)
axes[1].set(title="Observed and frozen-tape margins",
            xlabel="De-identified episode order", ylabel="Final coin margin")
axes[1].legend()
plt.tight_layout()
plt.show()

moon_transitions.to_frame("episodes")

Moon V9 converted 32 of 34 losses and preserved all 36 wins in the 70-game frozen-tape snapshot. That looked excellent until live cross-play exposed a consistent blind spot.

In [4]:
moon_live = moon_crossplay.sort_values("mean_margin")
fig, ax = plt.subplots(figsize=(9.5, 5.2))
colors = np.where(moon_live["mean_margin"].ge(0), "#2f855a", "#c53030")
ax.barh(moon_live["opponent"], moon_live["mean_margin"], color=colors)
ax.axvline(0, color="#222222", linewidth=1)
ax.set_xscale("symlog", linthresh=5_000)
ax.set(title="Moon V9 live cross-play — two unseen seeds and both seats",
       xlabel="Mean final coin margin", ylabel="Public policy")
plt.tight_layout()
plt.show()

moon_live[["opponent", "wins", "losses", "mean_margin", "worst_margin"]].reset_index(drop=True)

It lost all four games against each of Kaito V18, Beicicc C18, and Hamburger. The disagreement between frozen-tape coverage and live interaction is the central selection result of this notebook.

### 3. Beicicc C18 passes both gates

In [5]:
c18_counterfactual["transition"] = (
    c18_counterfactual["result_before"] + "→" + c18_counterfactual["result_after"]
)
c18_transitions = (
    c18_counterfactual["transition"].value_counts()
    .reindex(["win→win", "loss→win", "loss→loss", "win→loss"], fill_value=0)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
c18_transitions.plot.bar(
    ax=axes[0], color=["#2f855a", "#38a169", "#c53030", "#9b2c2c"]
)
axes[0].set(title="Beicicc C18 on 71 frozen tapes",
            xlabel="", ylabel="Episodes")
axes[0].tick_params(axis="x", rotation=0)

strong_live = c18_crossplay.sort_values("mean_margin")
bars = axes[1].barh(strong_live["opponent"], strong_live["mean_margin"], color="#2b6cb0")
axes[1].axvline(0, color="#222222", linewidth=1)
axes[1].set_xlim(0, strong_live["mean_margin"].max() * 1.22)
axes[1].set(title="Live cross-play against the strongest public family",
            xlabel="Mean final coin margin", ylabel="Public policy")
for bar, (_, row) in zip(bars, strong_live.iterrows()):
    label = f'{int(row["wins"])}W/{int(row["ties"])}T/{int(row["losses"])}L'
    axes[1].text(bar.get_width() + 70, bar.get_y() + bar.get_height() / 2,
                 label, va="center", fontsize=9)
plt.tight_layout()
plt.show()

summary = {
    "frozen_tape_games": int(len(c18_counterfactual)),
    "wins_after": int(c18_counterfactual["result_after"].eq("win").sum()),
    "loss_to_win": int((
        c18_counterfactual["result_before"].eq("loss")
        & c18_counterfactual["result_after"].eq("win")
    ).sum()),
    "win_to_loss": int((
        c18_counterfactual["result_before"].eq("win")
        & c18_counterfactual["result_after"].eq("loss")
    ).sum()),
    "live_wins": int(c18_crossplay["wins"].sum()),
    "live_ties": int(c18_crossplay["ties"].sum()),
    "live_losses": int(c18_crossplay["losses"].sum()),
}
summary

Beicicc C18 converted all 35 baseline losses and preserved all 36 baseline wins on the updated frozen-tape corpus. It then recorded **22 wins, 2 ties, and 0 losses** across 24 live games against Kaito V18, Hamburger, Moon V9, and Soil V8, using three unseen seeds and both `player` positions.

This does not prove a future leaderboard score. It does show that the candidate survived both the historical compatibility gate and the live-interaction gate used here.

## Public References

The policies named in the comparison are public Kaggle notebooks. No source code or action trace from them is reproduced here.

- [Kaito V18 Closed Loop](https://www.kaggle.com/code/kaitofukami/40-53-top-10-future-holdout-v18-closed-loop)
- [Beicicc C18 Online Entrypoint Safe](https://www.kaggle.com/code/beicicc/kaggriculture-c18-online-entrypoint-safe)
- [The Moon Counts Melons](https://www.kaggle.com/code/prvsiyan/kaggriculture-frontier-the-moon-counts-melons)
- [The Soil Remembers Rain](https://www.kaggle.com/code/prvsiyan/kaggriculture-frontier-the-soil-remembers-rain)
- [Kaggriculture Hamburger](https://www.kaggle.com/code/romantamrazov/kaggriculture-hamburger)

Analysis and plotting code authored in this notebook is released under the Apache License 2.0. Linked notebooks retain their own stated licenses and attribution.

## Takeaways

1. **Prioritize small-gap, high-similarity losses.** They isolate marginal economic decisions better than route-level blowouts.
2. **Separate compatibility from interaction.** A policy can dominate frozen replay tapes and still lose consistently when opponents react live.
3. **Do not stack unvalidated overlays.** Promote one coherent route or one narrow guard at a time so that gains remain attributable.
4. **Use both seats and unseen seeds.** Seat-coherent schedules and shared-market interactions can otherwise produce misleading conclusions.
5. **Keep claims bounded.** These experiments select a robust research candidate; they do not forecast a leaderboard rating.